# 1D elastic collisions: does a gas thermalize?

**Goal.** Explore the evolution of an initially non-equilibrium velocity distribution in a homogeneous, spatially infinite, one-dimensional gas. Compare equal-mass particles with a binary mixture.
For a collision between particles of masses $m_i,m_j$, define $\alpha_{ij}=(m_i-m_j)/(m_i+m_j)$. Then

$$v_i^{\prime}=\alpha_{ij}v_i+(1-\alpha_{ij})v_j,\qquad v_j^{\prime}=(1+\alpha_{ij})v_i-\alpha_{ij}v_j.$$

For identical masses ($\alpha=0$), velocities are simply exchanged. In a mixture, the pairwise $\alpha_{ij}$ changes sign depending on collision order; collisions between particles belonging to the same species always have $\alpha=0$.

**Time model.** The gas occupies a periodic and homogeneous segment of length $L$ (no boundaries are simulated). Each unordered pair collides with rate $\sigma |v_i-v_j|/L$. The Monte Carlo approach is used to simulate the time evolution of the gas. This statistical technique samples the next pair with probability proportional to its rate and the time increment from an exponential distribution with the sum of rates.

The Gaussian comparison has the same mean velocity and total kinetic energy as the simulated gas. Define the total mass, momentum and kinetic energy as $M=\sum_i m_i$, $P=\sum_i m_i v_i$, and $E=\tfrac12\sum_i m_i v_i^2$. The centre-of-mass velocity and the reference temperature are then

$$u=\frac{P}{M},\qquad E-\frac12Mu^2=\frac{N}{2}k_B T,\qquad T=\frac{2E-Mu^2}{N k_B}. $$

The code uses $k_B=1$. For a Maxwellian mixture, each species has $\left\langle(v-u)^2\right\rangle_a=k_B T/m_a$. A separate mass-dependent Gaussian is plotted for each species using this common temperature. This Gaussian (Maxwellian) is taken as a **reference**, despite this special 1D system not being guaranteed to converge to this distribution.


In [6]:
import numpy as np
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import display

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    import piplite
    await piplite.install('ipywidgets==8.1.7')
    import ipywidgets as widgets

plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .2})

In [7]:
def initial_velocities(rng, n, kind):
    if kind == "Uniform":
        return rng.uniform(-2, 2, n)
    if kind == "Gaussian":
        return rng.normal(0, 1, n)
    if kind == "Monokinetic":
        return np.ones(n)  # no relative velocity: no collisions
    if kind == "Two beams":
        v = np.ones(n)
        v[:n//2] = -1
        rng.shuffle(v)
        return v
    if kind == "Random discrete":
        return rng.choice([-2., -0.5, 0.5, 2.], n)
    raise ValueError(kind)


def simulate(n=120, alpha=0.0, fraction_heavy=.5, sigma=1., length=120.,
             events=2500, initial="Two beams", seed=1, frames=40):
    if not (-1 < alpha < 1):
        raise ValueError("alpha must be strictly between -1 and 1")
    if n < 2 or sigma <= 0 or length <= 0:
        raise ValueError("Invalid population or collision parameters")
    rng = np.random.default_rng(seed)
    # alpha = (m_light - m_heavy)/(m_light + m_heavy) for a light-heavy pair
    # For alpha > 0, the second species is lighter. Both signs are supported.
    mass_ratio = (1-alpha)/(1+alpha)
    masses = np.ones(n) # initialize all particles with mass 1

    if abs(alpha) > 1e-12:
        count = min(n-1, max(1, int(round(n*fraction_heavy))))
        masses[:count] = mass_ratio # assign the other mass to the first "count" particles
        rng.shuffle(masses) # shuffle the masses to avoid any ordering bias

    v = initial_velocities(rng, n, initial)
    p0 = np.dot(masses, v)
    e0 = .5*np.dot(masses, v*v)
    ii, jj = np.triu_indices(n, k=1) # indices of all unique pairs (i,j) with i<j
    checkpoints = np.unique(np.linspace(0, events, frames+1, dtype=int))
    snapshots = [v.copy()]

    times = [0.]
    steps = [0]
    t = 0.
    for step in range(1, events+1):
        rates = sigma/length * np.abs(v[ii]-v[jj])
        total = rates.sum()
        if total <= 0:
            break
        t += rng.exponential(1/total) # time until next collision (sampled from exponential distribution)
        pick = np.searchsorted(np.cumsum(rates), rng.random()*total)
        i, j = ii[pick], jj[pick]
        a = (masses[i]-masses[j])/(masses[i]+masses[j]) # compute current alpha for the selected pair
        vi, vj = v[i], v[j]
        # update post-collision velocities according to the elastic collision rules
        v[i] = a*vi + (1-a)*vj
        v[j] = (1+a)*vi - a*vj

        if step in checkpoints:
            snapshots.append(v.copy())
            times.append(t)
            steps.append(step)

    # Preserve the final state if the process terminated early
    if steps[-1] != step if events else False:
        snapshots.append(v.copy())
        times.append(t)
        steps.append(step)

    return dict(m=masses, snapshots=np.asarray(snapshots), times=np.asarray(times),
                steps=np.asarray(steps), p0=p0, e0=e0, sigma=sigma, length=length,
                alpha=alpha)


def reference_parameters(run):
    m = run['m']; n = len(m)
    u = run['p0']/m.sum()
    # E_cm = (N/2) k_B T for 1D equipartition, with k_B = 1
    temperature = max(0., (2*run['e0']-m.sum()*u*u)/n)
    return u, temperature


def normal_pdf(x, mean, std):
    if std <= 0: 
        return np.zeros_like(x)

    return np.exp(-.5*((x-mean)/std)**2)/(np.sqrt(2*np.pi)*std)

## Interactive simulation

Set **α = 0** for identical masses; a nonzero value creates a binary mixture with masses $m_A=1$ and $m_B=(1-\alpha)/(1+\alpha)$. Set the mixture fraction to control the fraction of species B. **Run simulation** recomputes the trajectory; the snapshot slider then explores the stored evolution without recomputation. The step counter is collision count, whereas $t$ is sampled physical time in the chosen units.

In [8]:
N = widgets.IntSlider(value=300, min=50, max=400, step=10, description='Particles', continuous_update=False)
alpha_w = widgets.FloatSlider(value=0., min=-.9, max=.9, step=.1, description='α', readout_format='.2f', continuous_update=False)
frac = widgets.FloatSlider(value=.5, min=.1, max=.9, step=.1, description='Species B', continuous_update=False)
sigma_w = widgets.FloatLogSlider(value=1., base=10, min=-1, max=1, step=.1, description='σ', continuous_update=False)
events_w = widgets.IntSlider(value=15000, min=1000, max=30000, step=100, description='Collisions', continuous_update=False)
init_w = widgets.Dropdown(options=['Two beams', 'Uniform', 'Gaussian', 'Monokinetic', 'Random discrete'], value='Two beams', description='Initial')
seed_w = widgets.IntText(value=1, description='Seed')
button = widgets.Button(description='Run simulation', button_style='primary')
frame = widgets.IntSlider(value=0, min=0, max=40, description='Snapshot', continuous_update=False)
play = widgets.Play(value=0, min=0, max=40, step=1, interval=180)
widgets.jslink((play,'value'),(frame,'value'))
plot_out = widgets.Image(format='png')
info = widgets.HTML()
state = {'run': None}

def render_figure(fig, target, dpi=110):
    buffer = BytesIO()
    fig.savefig(buffer, format='png', dpi=dpi, bbox_inches='tight')
    plt.close(fig)
    target.value = buffer.getvalue()

def draw(change=None):
    r = state['run']
    if r is None: return
    k = min(frame.value, len(r['steps'])-1)
    v = r['snapshots'][k]; m = r['m']; u, temp = reference_parameters(r)
    all_v = r['snapshots'].ravel()
    lo = min(all_v.min(), u-4*np.sqrt(temp/max(m.min(), 1e-8)))
    hi = max(all_v.max(), u+4*np.sqrt(temp/max(m.min(), 1e-8)))
    pad = .05*(hi-lo+1e-10)
    x = np.linspace(lo-pad,hi+pad,400)
    fig, ax = plt.subplots(figsize=(9,4.5))
    for val, label, color in [(1., 'Species A', 'C0'), ((1-r['alpha'])/(1+r['alpha']), 'Species B', 'C1')]:
        mask = np.isclose(m, val)
        if not np.any(mask): continue
        nbins = int(np.clip(1.5 * np.sqrt(mask.sum()), 16, 28))
        ax.hist(v[mask], bins=nbins, range=(lo-pad, hi+pad), density=True,
                alpha=.35, label=f'{label} (n={mask.sum()})', color=color)
        ax.plot(x, normal_pdf(x,u,np.sqrt(temp/val)), color=color, lw=2,
                label=f'{label}: Gaussian reference')
        if abs(r['alpha']) < 1e-12: break
    ax.set(xlabel='Velocity v', ylabel='Probability density',
           title=f'Collision {r["steps"][k]:,}   |   t = {r["times"][k]:.4g}')
    ax.legend(fontsize=8)
    render_figure(fig, plot_out)
    p = np.dot(m,v); e = .5*np.dot(m,v*v)
    info.value = (f'<b>Momentum error:</b> {abs(p-r["p0"]):.2e} &nbsp; '
                  f'<b>Energy error:</b> {abs(e-r["e0"]):.2e} &nbsp; '
                  f'<b>Reference temperature (kB=1):</b> {temp:.4g}')

def run_clicked(_=None):
    r = simulate(n=N.value, alpha=alpha_w.value, fraction_heavy=frac.value,
                 sigma=sigma_w.value, events=events_w.value,
                 initial=init_w.value, seed=seed_w.value)
    state['run'] = r
    frame.max = len(r['steps'])-1
    play.max = frame.max
    frame.value = 0

display(widgets.VBox([
    widgets.HBox([N, alpha_w, frac]),
    widgets.HBox([sigma_w, events_w, init_w]),
    widgets.HBox([seed_w, button]),
    widgets.HBox([play, frame]), info, plot_out
]))
run_clicked()
draw()

## Macroscopic observables and coarse-grained entropy

The simulator uses an **effective one-dimensional normalization length** $L$, not a physical 3D volume. Hence the following are **linear densities**: $n=N/L$, $\mathcal P=\sum_i m_i v_i/L$, and $\mathcal E=\sum_i m_i v_i^2/(2L)$. They remain constant for elastic collisions. To interpret these as volumetric densities, replace the collision-rate normalization $\sigma_{1D}/L$ consistently with $\sigma_{3D}/V$ throughout the model.

For each species $a$ and velocity bin $k$, let $N_{ak}$ be the number of particles of species $a$ in bin $k$, $N_a$ the total number of particles of that species, and $\Delta v$ the common bin width. The index $k$ is a **bin index**. 
The normalized, piecewise-constant velocity distribution is estimated as

$$f_{a,k}=\frac{N_{ak}}{N_a\,\Delta v},\qquad \sum_k f_{a,k}\Delta v=1. $$

The plotted coarse-grained entropy per particle is

$$s_{\mathrm{cg}}=-\sum_a\frac{N_a}{N}\sum_k f_{a,k}\,\Delta v\ln f_{a,k}. $$

This is a species-resolved histogram estimate of the differential entropy, averaged across species. Strictly, the argument of a logarithm must be dimensionless: one may replace $\ln f_{a,k}$ with $\ln(f_{a,k}/f_0)$, where $f_0$ is a fixed reference density. That only adds a constant, so the code omits it and the displayed value should be read as an entropy **surrogate**. The mixing contribution for fixed species populations is also omitted. This is **not** the exact thermodynamic entropy. Finite-sample fluctuations and coarse-graining can make it non-monotonic. Fixed velocity-bin edges are essential for meaningful comparisons. For a delta-like distribution, the result depends strongly on bin width.

The common temperature shown in the last panel follows from $T=(2E-Mu^2)/(Nk_B)$, with $u=P/M$ and $k_B=1$ in the code.

The kinetic energy separates into centre-of-mass (bulk) and thermal parts as $E=\tfrac12Mu^2+E_{\rm th}$, with $E_{\rm th}=\tfrac12\sum_i m_i(v_i-u)^2$. In one velocity dimension, equipartition gives $E_{\rm th}=Nk_B T/2$: the factor $1/2$ is due to the single translational degree of freedom per particle.

The panels below also show species mean velocities. At a common Maxwellian equilibrium, the mean velocities approach $u$, the kinetic temperatures approach the common reference $T$. The vertical cursor marks the selected snapshot. The existing snapshot slider and Play button control all plots.

In [9]:
def macroscopic_observables(run, bins=45):
    velocities = run['snapshots']
    masses = run['m']
    length = run['length']
    nframes, n = velocities.shape
    number_density = np.full(nframes, n / length)
    momentum_density = velocities @ masses / length
    energy_density = 0.5 * (velocities**2) @ masses / length

    # Fixed bin edges over all saved states and both species.
    vmin, vmax = velocities.min(), velocities.max()
    margin = max(0.05 * (vmax-vmin), 0.05)
    edges = np.linspace(vmin-margin, vmax+margin, bins+1)
    dv = edges[1]-edges[0]
    unique_masses = np.unique(masses)
    species_masks = [np.isclose(masses, mass) for mass in unique_masses]
    entropy = np.zeros(nframes)
    species_temperatures = np.empty((nframes, len(species_masks)))
    species_mean_velocities = np.empty((nframes, len(species_masks)))

    for a, mask in enumerate(species_masks):
        vv = velocities[:, mask]
        mean_v = np.mean(vv, axis=1)
        centred_v = vv - mean_v[:, None]
        variance_v = np.mean(centred_v**2, axis=1)
        species_mean_velocities[:, a] = mean_v
        species_temperatures[:, a] = masses[mask][0] * variance_v
        weight = mask.sum() / n
        for k, row in enumerate(vv):
            counts, _ = np.histogram(row, bins=edges)
            f = counts[counts > 0] / (mask.sum() * dv)
            entropy[k] -= weight * np.sum(f * np.log(f) * dv)

    return dict(n=number_density, momentum=momentum_density,
                energy=energy_density, entropy=entropy,
                species_temperatures=species_temperatures,
                species_mean_velocities=species_mean_velocities,
                species_masses=unique_masses, bin_edges=edges)


In [10]:
macro_out = widgets.Image(format='png')

def draw_macros(change=None):
    run = state['run']
    if run is None:
        return
    obs = macroscopic_observables(run)
    t = run['times']
    k = min(frame.value, len(t)-1)
    fig, axes = plt.subplots(3, 3, figsize=(13, 10), constrained_layout=True)
    panels = [
        ('Number density n', obs['n'], 'n'),
        ('Momentum density', obs['momentum'], 'P'),
        ('Kinetic energy density', obs['energy'], 'E'),
        ('Coarse-grained entropy per particle', obs['entropy'], 's*'),
    ]
    for ax, (title, values, ylabel) in zip(axes.flat, panels):
        ax.plot(t, values, lw=1.8)
        ax.axhline(values[0], color='gray', ls='--', lw=1, label='Initial value')
        ax.axvline(t[k], color='black', ls=':', lw=1)
        ax.scatter([t[k]], [values[k]], s=35, zorder=5)
        ax.set(title=title, xlabel='Time', ylabel=ylabel)
        ax.ticklabel_format(axis='y', style='sci', scilimits=(-3, 4), useOffset=False)
        if ylabel == 'P':
            ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=4))
            ax.yaxis.set_major_formatter(plt.FormatStrFormatter('%.2g'))

    u, reference_temperature = reference_parameters(run)
    ax = axes.flat[4]
    for a, mass in enumerate(obs['species_masses']):
        ax.plot(t, obs['species_temperatures'][:, a], label=f'm={mass:.3g}')
    ax.axhline(reference_temperature, color='gray', ls='--', lw=1.2, label='Reference T')
    ax.axvline(t[k], color='black', ls=':', lw=1)
    ax.set(title='Species kinetic temperatures (kB=1)', xlabel='Time', ylabel='T')
    ax.legend(fontsize=8)

    ax = axes.flat[5]
    for a, mass in enumerate(obs['species_masses']):
        ax.plot(t, obs['species_mean_velocities'][:, a], label=f'm={mass:.3g}')
    ax.axhline(u, color='gray', ls='--', lw=1.2, label='Centre-of-mass velocity')
    ax.axvline(t[k], color='black', ls=':', lw=1)
    ax.set(title='Species mean velocities', xlabel='Time', ylabel='mean v')
    ax.legend(fontsize=8)


    # Replace the unused bottom row with one centred information box.
    grid_spec = axes[2, 0].get_subplotspec().get_gridspec()
    for bottom_ax in axes[2, :]:
        bottom_ax.remove()
    ax = fig.add_subplot(grid_spec[2, :])
    ax.axis('off')
    ax.text(.5, .5, f'Collision events: {run["steps"][k]:,}\n'
                  f'Time: {t[k]:.5g}\n'
                  f'Mean collisions/particle: {2*run["steps"][k]/len(run["m"]):.3g}\n\n'
                  f'Entropy change: {obs["entropy"][k]-obs["entropy"][0]:+.4g}',
            va='center', ha='center', fontsize=11, transform=ax.transAxes)

    render_figure(fig, macro_out)

def draw_all(change=None):
    draw(change)
    draw_macros(change)

def run_and_draw_all(_=None):
    # Avoid an intermediate redraw when run_clicked resets the frame to zero.
    frame.unobserve(draw_all, names='value')
    run_clicked()
    frame.observe(draw_all, names='value')
    draw_all()

# A re-executed cell must not accumulate observers or button callbacks.
frame.unobserve_all('value')
for callback in list(button._click_handlers.callbacks):
    button.on_click(callback, remove=True)
frame.observe(draw_all, names='value')
button.on_click(run_and_draw_all)
display(macro_out)
draw_macros()


Image(value=b'')

## Suggested experiments

1. **Equal masses:** choose $\alpha=0$, *Two beams*, and many collisions. The velocity histogram is invariant: each collision only exchanges the two velocities.
2. **Monokinetic:** set *Monokinetic*. All relative velocities vanish; the collision rate is zero, irrespective of $\alpha$.
3. **Cross section:** rerun with the same seed and different $\sigma$. The asymptotic state is unchanged, but the physical time scale changes approximately as $1/\sigma$.

**Model limitations.** This is a finite, homogeneous 1D elastic-particle system with stochastic collision pairing. It does not automatically reproduce molecular chaos or a three-dimensional Boltzmann gas. In particular, the equilibrium theoretical reference is used as a reference diagnostic towards thermalisation.